# 랭체인 에이전트

- https://python.langchain.com/docs/tutorials/agents/#installation

- https://python.langchain.com/docs/integrations/tools/

### 빌트인 도구
- 랭체인에서 제공하는 사전에 정의된 도구와 툴킷
- 툴은 단일 도구, 툴킷은 여러 도구를 묶어서 하나의 도구로 활용 가능
- Langchain Tools/toolkit


## 1. 검색을 하는 툴 사용해보기

### Tavily

- 대형 언어 모델(LLM)과 AI 에이전트를 위해 설계된 고성능 검색 엔진이자 API 플랫폼
- AI가 실시간으로 정확하고 사실에 기반한 정보를 검색하고 활용할 수 있도록 돕는 도구
- https://www.tavily.com/

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
%pip install -qU langchain langchain-openai langchain-tavily tiktoken langchain-community

# uv add langchain langchain-openai langchain-tavily tiktoken langchain-community

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

# 웹 서치 툴 : 최신 뉴스/웹 결과
from langchain_tavily import TavilySearch


In [4]:
tavily = TavilySearch()         # TAVILY_API_KEY

In [5]:
# 시스템 프롬프트
system_prompt = '너는 툴만 사용해서 사실을 확인하고 한국어로 간략히 답한다.'

In [ ]:
llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0
)

agent = create_agent(
    model=llm,
    tools=[tavily],
    system_prompt=system_prompt
)

In [7]:
# 실행
result = agent.invoke({
    'messages':[{'role':'user', 'content':'하이닉스의 주요 뉴스 3개 요약해줘.'}]
})

In [8]:
result

{'messages': [HumanMessage(content='하이닉스의 주요 뉴스 3개 요약해줘.', additional_kwargs={}, response_metadata={}, id='adf3626d-b664-46e5-b5fa-0b1efabc2da0'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 1223, 'total_tokens': 1258, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_9865f4ebfa', 'id': 'chatcmpl-DtUO2S65F6cbmIBjsbof9ylzVbJQ5', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eee76-b602-7810-ad90-0ba9b9f0d31d-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '하이닉스 주요 뉴스', 'search_depth': 'advanced', 'topic': 'news', 'time_range': 'month'}, 'id': 'call_aIgFl00ej915QME6ZeLY3vJC', 'type': 'tool_cal

In [9]:
print(result['messages'][-1].content)

최근 한 달간 하이닉스 관련 주요 뉴스 3가지를 요약하면 다음과 같습니다.

1. 엔비디아와 하이닉스가 AI용 메모리 칩 공동 개발을 위한 다년간 파트너십 계약을 체결했습니다. 이로 인해 하이닉스 주가는 변동성을 보였으나 AI 수요 증가로 메모리 칩 공급 부족 현상이 수년간 지속될 전망입니다.

2. 엔비디아 CEO 젠슨 황은 한국 방문 중 하이닉스와 삼성전자 등과 협력해 AI 및 로보틱스 분야에서 미래 협력을 강화할 계획임을 밝혔습니다. 한국은 대규모 제조 인프라를 갖춘 AI 상용화 테스트베드로 부상하고 있습니다.

3. AI 칩 수요 급증으로 하이닉스와 삼성전자, 마이크론 등 주요 메모리 업체들이 1조 달러 이상의 시가총액을 기록하며 AI 메모리 칩 시장에서 강력한 가격 결정력을 확보하고 있습니다.


In [10]:
tavily2 = TavilySearch(max_results=3)        # 검색 결과 3개로 제한

system_prompt2 = '너는 검색 툴만 사용해서 최신 정보를 확인하고 한국어로 5줄 이내로 요약한다.'

agent2 = create_agent(
    model=llm,
    tools=[tavily2],
    system_prompt=system_prompt2
)

In [11]:
q = 'AI 관련 최신 뉴스 핵심만 알려줘'
result2 = agent2.invoke({
    'messages':[{'role':'user', 'content':q}]
})

print(result2['messages'][-1].content)

최근 AI 뉴스 핵심 요약입니다. 
1. Peec AI가 AI 쇼핑 분석 도구를 출시해 전자상거래 브랜드가 AI 추천 제품을 실시간으로 추적할 수 있게 됐습니다.
2. 미국 트럼프 행정부는 명확한 AI 규칙 없이 국가 안보와 경제 영향 중심의 그림자 AI 정책을 추진 중입니다.
3. 인도에서는 AI 인력이 비기술 배경과 여성, 2급 도시에서 빠르게 증가하며 AI 업스킬링으로 평균 147% 급여 상승 효과가 나타나고 있습니다.


In [12]:
query = "오늘 한국 AI 반도체 관련 최신 뉴스 5개만 핵심 요약해줘."

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    latest_message = chunk["messages"][-1]

    # 1. 모델이 툴 호출을 결정한 경우
    if getattr(latest_message, "tool_calls", None):
        for tool_call in latest_message.tool_calls:
            print(f"\n[툴 호출] {tool_call['name']}")
            print(f"[툴 입력] {tool_call['args']}")

    # 2. 툴 실행 결과
    elif latest_message.type == "tool":
        print(f"\n[툴 결과] {latest_message.content[:1000]}")

    # 3. 최종 응답
    elif latest_message.content:
        print(f"\n[에이전트 응답]\n{latest_message.content}")


[에이전트 응답]
오늘 한국 AI 반도체 관련 최신 뉴스 5개만 핵심 요약해줘.

[툴 호출] tavily_search
[툴 입력] {'query': '한국 AI 반도체 최신 뉴스', 'time_range': 'day', 'search_depth': 'basic', 'topic': 'news'}

[툴 결과] {"query": "한국 AI 반도체 최신 뉴스", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.reuters.com/world/asia-pacific/sk-hynix-overtakes-samsung-become-koreas-most-valuable-company-2026-06-22/", "title": "SK Hynix overtakes Samsung to become Korea's most valuable company - Reuters", "score": 0.53997856, "published_date": "Mon, 22 Jun 2026 05:51:19 GMT", "content": "The company, now the dominant supplier of high-bandwidth memory (HBM) chips used in AI systems for customers such as ‌Nvidia (NVDA.O), opens new tab and Alphabet's Google (GOOGL.O), opens new tab, has emerged as one of the biggest beneficiaries of the global AI boom, propelling a more than 340% rally in its shares this year and lifting its market value above both Samsung Electronics and Micron (MU.O), opens new tab.\n\n

## 2. 파일 저장하는 툴 사용해보기

In [13]:
# agent_filesystem_news.py
import os, datetime, csv
from pathlib import Path
from typing import List, Dict

In [14]:
# 파일 툴 정의에 사용할 LangChain v1 tool 데코레이터
from langchain.tools import tool

In [15]:
# --- 샌드박스 디렉터리(여기 밖은 접근 불가) ---
ROOT = Path("../01_agent_result").absolute()
ROOT.mkdir(parents=True, exist_ok=True)

In [16]:
# 이전 버전에서는 이렇게 사용
# # 파일 툴킷 : 읽기/쓰기/목록
# from langchain_community.agent_toolkits import FileManagementToolkit\

# # 파일 관리 툴킷 (read_file / write_file / list_directory 등 제공)
# file_tools = FileManagementToolkit(root_dir=str(ROOT)).get_tools()
# # 참고: 이 툴킷은 루트 디렉터리 내에서 Copy/Move/Delete/Read/Write/List 등을 제공. :contentReference[oaicite:1]{index=1}

In [17]:
# 파일 관리 도구 직접 정의 (read_file / write_file / list_directory)
def _safe_path(path: str) -> Path:
    """ROOT 밖으로 나가지 않도록 경로를 검증합니다."""
    target = (ROOT / (path or ".")).resolve()
    root = ROOT.resolve()
    if root != target and root not in target.parents:
        raise ValueError(f"ROOT 밖의 경로는 사용할 수 없습니다: {path}")
    return target

@tool
def read_file(file_path: str) -> str:
    """Read a UTF-8 text file under the sandbox root directory."""
    return _safe_path(file_path).read_text(encoding="utf-8")

@tool
def write_file(file_path: str, text: str) -> str:
    """Write UTF-8 text to a file under the sandbox root directory."""
    target = _safe_path(file_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding="utf-8")
    return f"saved: {target.relative_to(ROOT.resolve()).as_posix()}"

@tool
def list_directory(directory_path: str = ".") -> str:
    """List files and directories under the sandbox root directory."""
    target = _safe_path(directory_path)
    if not target.exists():
        return f"not found: {directory_path}"
    if target.is_file():
        return target.relative_to(ROOT.resolve()).as_posix()
    entries = []
    for item in sorted(target.iterdir(), key=lambda p: p.name):
        rel = item.relative_to(ROOT.resolve()).as_posix()
        entries.append(rel + ("/" if item.is_dir() else ""))
    return "\n".join(entries) if entries else "(empty)"

file_tools = [read_file, write_file, list_directory]


In [18]:
# 뉴스/웹 검색 툴
tavily = TavilySearch(max_results=5)

In [19]:
# --- 프롬프트 ---
SYSTEM = """
너는 '파일 시스템 툴'과 '웹 검색 툴'만 사용하는 어시스턴트다.
규칙
1) 최신 뉴스/링크는 Tavily로 검색한다.
2) 사용자가 CSV 저장을 요청하면, 다음 필드 헤더로 CSV를 만든다.:
    data, query, title, url, source, snippet
3) 파일 경로는 사용자가 지정하지 않으면 01_agent_result/new_{YYYYMMDD}.csv 로 저장한다.
4) 같은 날짜 파일이 이미 있더라도 새로 '덮어쓰기'하지 말고, 기존 내용을 읽어 헤더를 유지한 채 뒤로 이어붙인 텍스트를 만들고, write_file로 한번에 저장한다.
5) 저장 뒤에는 list_directory 또는 read_file을 활용해 저장 확인(행수/상위 3~5행) 정보를 보여준다.
6) 샌드박스 디렉터리(Root) 밖의 경로는 절대 사용하지 않는다.
모든 답변은 한국어로 간결하게.
"""

llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0)
TOOLS = [tavily] + file_tools

agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=SYSTEM
)

In [20]:
from datetime import datetime
def get_today():
    return datetime.now().strftime("%Y%m%d")

In [21]:
print("\n=== A. 뉴스 요약 ===")
user_input = f"{get_today()} 오늘 한국 AI 반도체 관련 최신 뉴스 5개만 핵심 요약해줘."
result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
print(result["messages"][-1].content)



=== A. 뉴스 요약 ===
아래는 **2026-06-22(오늘 기준)** “한국 AI 반도체” 관련 최신 이슈 **5개**를 핵심만 요약한 것입니다.

1) **SK하이닉스, 삼성전자 제치고 시총 1위(한국 최대 기업 등극)**
- **핵심:** SK하이닉스가 HBM(고대역폭 메모리) 강세를 바탕으로 시가총액이 삼성전자를 추월.
- **의미:** HBM이 AI 인프라의 핵심 부품으로 자리잡으며, 공급자 주도권이 강화되는 흐름.

2) **HBM 시장에서 SK하이닉스의 우위(점유율 확대)**
- **핵심:** AI용 HBM 수요 증가 속에서 SK하이닉스가 글로벌 HBM 시장에서 높은 점유율을 유지/확대.
- **의미:** AI 학습·추론용 메모리 병목을 HBM이 해결하면서, 한국 메모리 산업의 수혜가 지속.

3) **(투자/업황 관점) AI 반도체 랠리 모멘텀 점검 포인트로 ‘Micron 실적’ 주목**
- **핵심:** AI 랠리 흐름에서 메모리 업황(특히 HBM/DRAM 관련) 신호를 주는 이벤트로 Micron 실적이 언급.
- **의미:** 한국 기업에도 간접적으로 수요·가격·투자 사이클 전망에 영향.

4) **HBM이 ‘AI 시스템’과 결합되며 진입장벽 상승**
- **핵심:** 단순 메모리가 아니라 AI 프로세서와 촘촘히 결합되는 구조(적층 등) 때문에 공급 경쟁이 더 어려워지는 경향.
- **의미:** 특정 업체의 기술·생산 역량이 제품 포트폴리오에서 결정적 요인이 될 가능성.

5) **한국 기업들의 AI 인프라(메모리/반도체) 투자 정당화 강화**
- **핵심:** 글로벌 AI 붐이 메모리(HBM 포함) 밸류체인 전반의 투자 논리를 강화.
- **의미:** 단기 실적뿐 아니라 중장기 공급능력/공정 경쟁력 강화로 이어질 가능성.

원하시면 위 5개 각각에 대해 **(1) 한국 기업 관점 정리(수혜/리스크)**, **(2) 투자자 체크포인트(실적·CAPEX·수율·HBM 라인)** 형태로 더 압축해드릴게요.


In [22]:
result

{'messages': [HumanMessage(content='20260622 오늘 한국 AI 반도체 관련 최신 뉴스 5개만 핵심 요약해줘.', additional_kwargs={}, response_metadata={}, id='db17ae55-4df2-434f-b71b-3798e43bd9b6'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 1611, 'total_tokens': 1666, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DtUOI420EQ55SSuz5Z4tzs4sHe0nW', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eee76-f53f-7691-8ec6-64e64689c499-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': '2026년 6월 22일 한국 AI 반도체 최신 뉴스 5개 핵심 요약', 'topic': 'news', 'search_depth': 'advanced', 'time_range': 'day'}, 'id': 'call_hCeynu6

In [23]:
print("\n=== B. 검색 후 CSV 저장 ===")
user_input = f"{get_today()} 한국 반도체 최신 뉴스 5개를 찾아서 csv로 저장하고, 저장된 파일 상위 5행도 보여줘."
result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
print(result["messages"][-1].content)



=== B. 검색 후 CSV 저장 ===
요청하신 **2026-06-22 한국 반도체( AI 메모리/HBM 등 포함) 최신 뉴스 5개**를 CSV로 저장했고, 저장된 파일의 **상위 5행**도 함께 보여드립니다.

- 저장 파일: `01_agent_result/new_20260622.csv`

## CSV 상위 5행
```csv
data, query, title, url, source, snippet
"20260622","한국 AI 반도체 최신 뉴스 5개","SK Hynix announces multi-year tech deal with Nvidia for AI factories - Reuters","https://www.reuters.com/business/media-telecom/sk-hynix-announces-multi-year-tech-deal-with-nvidia-ai-factories-2026-06-07/","Reuters","SK hynix가 Nvidia와 차세대 메모리 및 AI 데이터센터 관련 다년 기술 파트너십을 발표."
"20260622","한국 AI 반도체 최신 뉴스 5개","NVIDIA and SK hynix Announce Multiyear Technology Partnership to Advance Memory for AI Factories - NVIDIA Newsroom","https://nvidianews.nvidia.com/news/sk-hynix-ai-factory","NVIDIA Newsroom","NVIDIA가 SK hynix와 AI 팩토리용 차세대 메모리 공동 개발 및 설계·제조 가속을 위한 다년 협력 발표."
"20260622","한국 AI 반도체 최신 뉴스 5개","Nvidia strikes AI cloud, memory deals in Korea - Light Reading","https://www.lightreading.com/ai-machine-learning/nvidia-strikes-ai-cloud-memo

In [24]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": user_input}]},
    stream_mode="values",
):
    latest_message = chunk["messages"][-1]

    # 1. 모델이 툴 호출을 결정한 경우
    if getattr(latest_message, "tool_calls", None):
        for tool_call in latest_message.tool_calls:
            print(f"\n[툴 호출] {tool_call['name']}")
            print(f"[툴 입력] {tool_call['args']}")

    # 2. 툴 실행 결과
    elif latest_message.type == "tool":
        print(f"\n[툴 결과] {latest_message.content[:1000]}")

    # 3. 최종 응답
    elif latest_message.content:
        print(f"\n[에이전트 응답]\n{latest_message.content}")


[에이전트 응답]
20260622 한국 반도체 최신 뉴스 5개를 찾아서 csv로 저장하고, 저장된 파일 상위 5행도 보여줘.

[툴 호출] tavily_search
[툴 입력] {'query': '2026년 6월 한국 반도체 최신 뉴스 5개', 'search_depth': 'advanced', 'topic': 'news', 'time_range': 'week'}

[툴 결과] {"query": "2026년 6월 한국 반도체 최신 뉴스 5개", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.thelec.net/news/articleView.html?idxno=11381", "title": "Applied Materials Unveils SiN ALD and Molybdenum Etch Systems for 3D Semiconductors - thelec.net", "score": 0.73459625, "published_date": "Wed, 17 Jun 2026 09:29:37 GMT", "content": "Login\n Join\n Mobile\n UPDATE : 2026-06-17 18:33 (Wed)\n\n KOREAN\n\n \n\nApplied Materials Unveils SiN ALD and Molybdenum Etch Systems for 3D Semiconductors\n\nCharacter Size Settings\n\n가\n\nThe body content of the article will be changed to this letter size.\n\n스크롤 이동 상태바\n\n# Applied Materials Unveils SiN ALD and Molybdenum Etch Systems for 3D Semiconductors\n\n## Centura Spectra and Producer Selectra target 

In [25]:
print("\n=== C. 작업 디렉터리 목록 ===")
user_input = f"{get_today()} 내 작업 폴더 안 파일 목록을 보여줘"
result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
print(result["messages"][-1].content)



=== C. 작업 디렉터리 목록 ===
현재 샌드박스 루트(`.`)의 파일/폴더 목록입니다.

- `01_agent_result/`


In [26]:
# C) 디렉터리 내 파일 목록 보기
print("\n=== C. 작업 디렉터리 목록 ===")
user_input = "내 작업 폴더 안 파일 목록을 보여줘"
result = agent.invoke({"messages": [{"role": "user", "content": user_input}]})
print(result["messages"][-1].content)



=== C. 작업 디렉터리 목록 ===
작업 폴더(샌드박스 루트) 안 파일/폴더 목록은 아래와 같습니다.

- `01_agent_result/`


## 3. 웹검색 -> sqlite 저장 -> 확인까지 하는 툴 만들기

- SQLite : 설치 없이 사용할 수 있는 가벼운 파일 기반 데이터베이스
- 특징 요약
    - 서버가 필요 없음 (파일 1개로 DB 완성)
    - SQL 문법 사용 (표준 SQL 지원)
    - 속도가 빠르고 가볍다
    - 파이썬 등 언어에 내장되어 있음 (sqlite3 모듈)

In [27]:
import os, datetime, sqlite3
from pathlib import Path
from langchain.tools import tool
from langchain_tavily import TavilySearch


In [35]:
# --- 작업 디렉터리 & SQLite ---
ROOT = Path("../01_agent_result");
ROOT.mkdir(exist_ok=True, parents=True)
DB_PATH = ROOT / 'news.db'

In [36]:
def _connect_db():
    return sqlite3.connect(DB_PATH)

In [37]:
# --- 툴 구성: SQL + Tavily ---
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

def _ensure_news_table() -> None:
    with _connect_db() as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS news (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                date TEXT NOT NULL,
                query TEXT NOT NULL,
                title TEXT NOT NULL,
                url TEXT NOT NULL UNIQUE,
                source TEXT,
                snippet TEXT
            )
            """
        )
        conn.commit()

@tool
def save_news_item(date: str, query: str, title: str, url: str, source: str = "", snippet: str = "") -> str:
    """Save one news item to the local SQLite news table using safe parameter binding."""
    _ensure_news_table()
    with _connect_db() as conn:
        cursor = conn.execute(
            """
            INSERT OR IGNORE INTO news (date, query, title, url, source, snippet)
            VALUES (?, ?, ?, ?, ?, ?)
            """,
            (date, query, title, url, source or "", snippet or ""),
        )
        conn.commit()
    if cursor.rowcount == 0:
        return f"duplicate skipped: {url}"
    return f"saved: {title}"

@tool
def list_tables() -> str:
    """List table names in the SQLite database."""
    _ensure_news_table()
    with _connect_db() as conn:
        rows = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        ).fetchall()
    return "\n".join(row[0] for row in rows) if rows else "(no tables)"

@tool
def describe_table(table_name: str) -> str:
    """Show columns and types for a SQLite table."""
    _ensure_news_table()
    with _connect_db() as conn:
        rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    if not rows:
        return f"table not found: {table_name}"
    return "\n".join(f"{row[1]} {row[2]}" for row in rows)

@tool
def run_sql(query: str) -> str:
    """Run SELECT/PRAGMA/WITH queries against the local SQLite database and return results."""
    _ensure_news_table()
    statement = query.strip().rstrip(";")
    if not statement:
        return "empty query"

    lowered = statement.lstrip().lower()
    if not lowered.startswith(("select", "pragma", "with")):
        return "run_sql은 조회 전용입니다. 저장은 save_news_item 도구를 사용하세요."

    try:
        with _connect_db() as conn:
            cursor = conn.execute(statement)
            rows = cursor.fetchall()
            columns = [desc[0] for desc in cursor.description or []]
    except sqlite3.Error as exc:
        return f"SQL error: {exc}"

    outputs = []
    if columns:
        outputs.append(" | ".join(columns))
    outputs.extend(" | ".join(str(value) for value in row) for row in rows)
    return "\n".join(outputs) if outputs else "(no rows)"

tavily = TavilySearch(max_results=5)
sql_tools = [save_news_item, list_tables, describe_table, run_sql]
Tools = [tavily] + sql_tools


In [38]:
# --- 프롬프트 ---
SYSTEM = """너는 '웹검색'과 'SQL 데이터베이스' 툴만 사용한다.
목표: 사용자가 원하면 최신 뉴스를 검색해서 SQLite(news.db)에 저장하고, 요청 시 조회/요약까지 수행한다.

[테이블 스키마]
테이블명: news
컬럼: id INTEGER PRIMARY KEY AUTOINCREMENT,
       date TEXT NOT NULL,         -- YYYY-MM-DD
       query TEXT NOT NULL,        -- 사용자가 요청한 검색어/명령
       title TEXT NOT NULL,
       url TEXT NOT NULL UNIQUE,   -- 중복 방지
       source TEXT,
       snippet TEXT

규칙:
1) 저장 전, news 테이블이 없는 경우 SQL로 직접 CREATE TABLE IF NOT EXISTS를 실행한다.
2) 웹 검색은 Tavily 툴로 수행하고 결과를 title/url/source/snippet으로 정규화한다.
3) date=오늘(한국시간 기준 YYYY-MM-DD), query=사용자 질의를 넣어 INSERT OR IGNORE로 저장한다.
4) 저장 후에는 방금 저장한 레코드 수와 상위 5건을 SELECT로 보여준다.
5) 조회만 요청하면 SELECT만 수행한다.
6) 항상 안전하고 간결한 SQL을 작성하고, 불필요한 전체 스캔/전체 컬럼 조회를 피한다.
7) 모든 답변은 한국어로 간결하게. 마지막에 한 줄 결론을 덧붙여라.
"""


In [39]:
agent3 = create_agent(
    model=llm,
    tools=Tools,
    system_prompt=SYSTEM
)

In [40]:
today = datetime.date.today().isoformat()
q = f"{today} 한국 AI 반도체 최신 뉴스 5개를 찾아 DB에 저장하고 상위 5건만 보여줘."
print("\n=== A. 검색 후 저장 & 확인 ===")
result = agent3.invoke({"messages": [{"role": "user", "content": q}]})
print(result["messages"][-1].content)



=== A. 검색 후 저장 & 확인 ===
2026-06-22 한국 AI 반도체 최신 뉴스 5건을 저장하고 상위 5건을 보여드립니다.

1. SK Hynix가 AI 시스템용 고대역폭 메모리(HBM) 칩 시장에서 61% 점유율로 삼성과 마이크론을 제치고 한국에서 가장 가치 있는 기업이 되었다. (Reuters)
2. 홍콩 IPO 시장에서 AI 관련 기업들이 계속 주도권을 잡고 있으며, 산업 및 소비재 기업들도 시장에 진입하고 있다. (CNBC)
3. 투자자들은 마이크론의 실적을 AI 랠리 모멘텀의 건강 상태를 확인하는 지표로 보고 있다. (TradingView)
4. Marvell의 CTO는 AI 시장이 현재 해결해야 할 병목에 도달했다고 말하며, 광통신 등 신기술이 반도체 산업에 큰 변화를 주고 있다고 설명했다. (Ynetnews)
5. 중국의 새로운 오픈소스 AI 모델 GLM-5.2가 실리콘밸리에서 주목받고 있으며, 긴 코딩 작업에 강점을 가진 대형 언어 모델이다. (Business Insider)

최신 한국 AI 반도체 뉴스 5건을 성공적으로 저장했습니다.


In [ ]:
# B) 조회만 (예: 오늘 저장분 5건)
q2 = "오늘 저장된 뉴스 중 제목과 출처만 5건 보여줘"
print("\n=== B. 조회만 ===")
result = agent3.invoke({"messages": [{"role": "user", "content": q2}]})
print(result["messages"][-1].content)



=== B. 조회만 ===
오늘(2026-06-22) 저장된 뉴스 5건의 **제목/출처**입니다.

1) **SK Hynix announces multi-year tech deal with Nvidia for AI factories - Reuters** / **Reuters**  
2) **Nvidia strikes AI cloud, memory deals in Korea - Light Reading** / **Light Reading**  
3) **Samsung, LG shares rally ahead of Nvidia CEO meetings with Korean executives - Reuters** / **Reuters**  
4) **“구글, 차세대 AI칩 생산 일부 삼성에 맡긴다” - 조선일보** / **조선일보**  
5) **SK Hynix, Micron join $1tn club on surging AI demand - BBC** / **BBC**
